In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
from snowflake.snowpark import Session

session = get_active_session()

print(f"Current database: {session.get_current_database()}")
print(f"Current schema: {session.get_current_schema()}")

In [ ]:
session.sql("""
CREATE DATABASE IF NOT EXISTS ATLAS_PLATFORM_DB;
""").collect()

session.sql("USE DATABASE ATLAS_PLATFORM_DB;").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS ATLAS_FEATURE_STORE;").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS ATLAS_MODEL_REGISTRY;").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS ATLAS_MONITORING;").collect()

print("Created ATLAS_PLATFORM_DB with FEATURE_STORE, MODEL_REGISTRY, MONITORING schemas.")


In [ ]:
# Show the feature store tables for the screenshot
session.sql("SHOW TABLES IN SCHEMA ATLAS_PLATFORM_DB.ATLAS_FEATURE_STORE;").show()


In [ ]:
#session.sql("SHOW COLUMNS IN TABLE CUSTOMER_TX_FEATURES;").show()


In [ ]:
--SHOW COLUMNS IN TABLE CUSTOMER_TX_FEATURES;

In [ ]:
session.sql("USE SCHEMA ATLAS_FEATURE_STORE;").collect()

session.sql("""
CREATE OR REPLACE TABLE CUSTOMER_TX_FEATURES (
    CUSTOMER_ID         STRING,
    AS_OF_DATE          DATE,
    TX_30D_COUNT        NUMBER,
    TX_30D_AMOUNT       NUMBER(18,2),
    TX_30D_AVG_TICKET   NUMBER(18,2)
);
""").collect()

session.sql("""
INSERT INTO CUSTOMER_TX_FEATURES (CUSTOMER_ID, AS_OF_DATE, TX_30D_COUNT, TX_30D_AMOUNT, TX_30D_AVG_TICKET)
VALUES
  ('CUST_001', CURRENT_DATE(), 12, 1200.00, 100.00),
  ('CUST_002', CURRENT_DATE(), 5,   250.00,  50.00),
  ('CUST_003', CURRENT_DATE(), 30, 6000.00, 200.00);
""").collect()

df_features = session.table("CUSTOMER_TX_FEATURES")
df_features.show()


In [ ]:
# Use the correct schema
session.sql("USE SCHEMA ATLAS_MODEL_REGISTRY;").collect()

# 1) Create the MODEL_REGISTRY table
session.sql("""
CREATE OR REPLACE TABLE MODEL_REGISTRY (
    MODEL_NAME         STRING,
    MODEL_VERSION      STRING,
    STAGE              STRING,
    FRAMEWORK          STRING,
    TRAINING_DATA_REF  STRING,
    METRICS            VARIANT,
    CREATED_AT         TIMESTAMP_NTZ,
    CREATED_BY         STRING
);
""").collect()

# 2) Insert a row using SELECT (so we can use expressions)
session.sql("""
INSERT INTO MODEL_REGISTRY (
    MODEL_NAME,
    MODEL_VERSION,
    STAGE,
    FRAMEWORK,
    TRAINING_DATA_REF,
    METRICS,
    CREATED_AT,
    CREATED_BY
)
SELECT
    'fraud_detection_model'                                           AS MODEL_NAME,
    'v1'                                                              AS MODEL_VERSION,
    'PROD'                                                            AS STAGE,
    'xgboost'                                                         AS FRAMEWORK,
    'ATLAS_FEATURE_STORE.CUSTOMER_TX_FEATURES:2025-11-01'             AS TRAINING_DATA_REF,
    '{"auc": 0.94, "f1": 0.88}'::VARIANT                              AS METRICS,
    CURRENT_TIMESTAMP()                                               AS CREATED_AT,
    'matt.reinsch'                                                    AS CREATED_BY;
""").collect()

# 3) Inspect for screenshots
session.table("MODEL_REGISTRY").show()


In [ ]:
session.sql("USE SCHEMA ATLAS_MONITORING;").collect()

session.sql("""
CREATE OR REPLACE TABLE PREDICTION_LOG (
    EVENT_TIME    TIMESTAMP_NTZ,
    MODEL_NAME    STRING,
    MODEL_VERSION STRING,
    ENTITY_ID     STRING,
    SCORE         FLOAT,
    LABEL         FLOAT,
    BATCH_ID      STRING
);
""").collect()

session.sql("""
INSERT INTO PREDICTION_LOG (EVENT_TIME, MODEL_NAME, MODEL_VERSION, ENTITY_ID, SCORE, LABEL, BATCH_ID)
VALUES
  (DATEADD('day', -1, CURRENT_TIMESTAMP()), 'fraud_detection_model', 'v1', 'CUST_001', 0.91, 1.0, 'batch_1'),
  (DATEADD('day', -1, CURRENT_TIMESTAMP()), 'fraud_detection_model', 'v1', 'CUST_002', 0.20, 0.0, 'batch_1'),
  (DATEADD('day', -10, CURRENT_TIMESTAMP()), 'fraud_detection_model', 'v1', 'CUST_003', 0.45, 0.0, 'batch_2'),
  (DATEADD('day', -15, CURRENT_TIMESTAMP()), 'fraud_detection_model', 'v1', 'CUST_004', 0.35, 0.0, 'batch_2');
""").collect()

print("Inserted sample prediction logs.")
session.table("PREDICTION_LOG").show()


In [ ]:
drift_sql = """
WITH recent AS (
    SELECT AVG(SCORE) AS avg_score
    FROM ATLAS_PLATFORM_DB.ATLAS_MONITORING.PREDICTION_LOG
    WHERE MODEL_NAME = 'fraud_detection_model'
      AND MODEL_VERSION = 'v1'
      AND EVENT_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
),
baseline AS (
    SELECT AVG(SCORE) AS avg_score
    FROM ATLAS_PLATFORM_DB.ATLAS_MONITORING.PREDICTION_LOG
    WHERE MODEL_NAME = 'fraud_detection_model'
      AND MODEL_VERSION = 'v1'
      AND EVENT_TIME < DATEADD('day', -7, CURRENT_TIMESTAMP())
)
SELECT
    recent.avg_score   AS recent_avg_score,
    baseline.avg_score AS baseline_avg_score,
    (recent.avg_score - baseline.avg_score) AS score_delta
FROM recent, baseline
"""

# Run it
drift_df = session.sql(drift_sql)
drift_df.show()



In [ ]:
df = session.sql("""
    SELECT AI_COMPLETE(
        'llama3.1-8b',
        'Write a haiku about Snowflake.'
    ) AS response
""")

df.show()



In [ ]:
recent = 0.555
baseline = 0.40

prompt = f"""
The model's recent average prediction score is {recent:.3f},
baseline is {baseline:.3f}.
Explain if this drift is meaningful and whether retraining is warranted.
"""

df = session.sql(f"""
    SELECT AI_COMPLETE(
        'llama3.1-8b',
        $$ {prompt} $$
    ) AS ANALYSIS
""")

analysis = df.collect()[0]['ANALYSIS']
print(analysis)


In [ ]:
CREATE OR REPLACE PROCEDURE ATLAS_TRIGGER_RETRAIN()
RETURNS STRING
LANGUAGE SQL
AS
$$
DECLARE
    drift FLOAT DEFAULT 0;
BEGIN
    -- Assign drift value from DRIFT_VIEW
    SELECT score_delta INTO :drift
    FROM ATLAS_PLATFORM_DB.ATLAS_MONITORING.DRIFT_VIEW;

    IF (:drift > 0.15) THEN
        CALL START_RETRAINING_PIPELINE();
        RETURN 'Drift detected — retraining triggered.';
    ELSE
        RETURN 'No drift — model baseline stable.';
    END IF;

END
$$;


In [ ]:
CREATE OR REPLACE TASK ATLAS_DRIFT_TASK
WAREHOUSE = COMPUTE_WH
SCHEDULE = '1 HOUR'
AS CALL ATLAS_TRIGGER_RETRAIN();

In [ ]:
SHOW TASKS LIKE 'ATLAS_DRIFT_TASK';



In [ ]:
session.sql("USE DATABASE ATLAS_PLATFORM_DB;").collect()
session.sql("USE SCHEMA ATLAS_FEATURE_STORE;").collect()

session.sql("""
CREATE OR REPLACE TABLE CUSTOMER_BEHAVIOR_FEATURES (
    CUSTOMER_ID              STRING,
    AS_OF_DATE               DATE,
    LOGIN_30D_COUNT          NUMBER,
    FAILED_LOGIN_7D_COUNT    NUMBER,
    DEVICE_DIVERSITY_30D     NUMBER,      -- distinct devices in last 30 days
    MFA_CHALLENGE_30D_COUNT  NUMBER
);
""").collect()

session.sql("""
INSERT INTO CUSTOMER_BEHAVIOR_FEATURES (
  CUSTOMER_ID, AS_OF_DATE,
  LOGIN_30D_COUNT, FAILED_LOGIN_7D_COUNT,
  DEVICE_DIVERSITY_30D, MFA_CHALLENGE_30D_COUNT
)
VALUES
  ('CUST_001', CURRENT_DATE(), 45,  1, 2, 0),
  ('CUST_002', CURRENT_DATE(), 12,  3, 1, 1),
  ('CUST_003', CURRENT_DATE(), 80,  0, 3, 0);
""").collect()

df_behavior = session.table("CUSTOMER_BEHAVIOR_FEATURES")
df_behavior.show()
